In [2]:
import re
import time
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path("../raw")
HTML_DIR = BASE_DIR / "lok_sabha_html"
CANDIDATE_DIR = BASE_DIR / "lok_sabha_candidate_dfs"
MANIFEST_PATH = HTML_DIR / "html_manifest.csv"

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"const_id": str, "state_id": str}
)
manifest["year"] = pd.to_numeric(manifest["year"]).astype(int)
manifest["const_id"] = manifest["const_id"].str.strip()

targets = {
    (2019, "UTTAR PRADESH", "905"),
    (2014, "MAHARASHTRA", "336"),
    (2014, "UTTAR PRADESH", "431"),
}

selected = manifest[
    manifest.apply(
        lambda r: (
            int(r["year"]),
            str(r["state_ut"]).strip().upper(),
            str(r["const_id"]).strip(),
        ) in targets,
        axis=1,
    )
].copy()

found_targets = {
    (
        int(r["year"]),
        str(r["state_ut"]).strip().upper(),
        str(r["const_id"]).strip(),
    )
    for _, r in selected.iterrows()
}

assert found_targets == targets, (
    f"Manifest does not contain exactly the expected targets. "
    f"Missing: {targets - found_targets}"
)
assert len(selected) == 3, f"Expected 3 manifest rows, got {len(selected)}"

CANDIDATE_HREF_RE = re.compile(r"candidate.*\.php", re.I)
WINNER_RE = re.compile(r"\bWinner\b", re.I)

def header_key(text):
    return re.sub(r"[^a-z]", "", str(text).lower())

def find_column(headers, pattern):
    for i, header in enumerate(headers):
        if pattern.search(header_key(header)):
            return i
    return None

def get_cell_value(cells, index):
    if index is None or index >= len(cells):
        return None
    value = cells[index].get_text(" ", strip=True)
    return value or None

def get_image_url(cell, base_url):
    if cell is None:
        return None
    img = cell.find("img")
    if img is None:
        return None
    src = img.get("src") or img.get("data-src")
    return urljoin(base_url, src) if src else None

columns = [
    "year", "state", "state_id", "const", "const_id",
    "is_byelection", "bye_election_date", "election_date",
    "candidate", "candidate_id", "candidate_url", "is_winner",
    "party", "criminal_cases", "education", "age",
    "total_assets", "asset_link", "liabilities", "liabilities_link",
    "constituency_url", "html_path",
]

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
new_rows = []

try:
    for _, page in selected.iterrows():
        year = int(page["year"])
        state = str(page["state_ut"]).strip()
        const = str(page["const"]).strip()
        const_id = str(page["const_id"]).strip()
        url = str(page["link"]).strip()

        print(f"\nRendering: {year} | {state} | {const} ({const_id})")

        driver.get(url)

        WebDriverWait(driver, 30).until(
            lambda d: len(
                d.find_elements(
                    By.CSS_SELECTOR,
                    "table.w3-table.w3-bordered a[href*='candidate']"
                )
            ) > 0
        )

        soup = BeautifulSoup(driver.page_source, "lxml")
        page_count = 0

        for table in soup.find_all("table"):
            header_row = table.find("tr", recursive=False)
            if header_row is None:
                header_row = table.find("tr")
            if header_row is None:
                continue

            headers = [
                cell.get_text(" ", strip=True)
                for cell in header_row.find_all(
                    ["th", "td"], recursive=False
                )
            ]

            candidate_col = find_column(
                headers, re.compile(r"candidate")
            )
            assets_col = find_column(
                headers, re.compile(r"totalassets|assets")
            )

            if candidate_col is None or assets_col is None:
                continue

            party_col = find_column(headers, re.compile(r"party"))
            criminal_col = find_column(
                headers, re.compile(r"criminalcases|criminalcase")
            )
            education_col = find_column(
                headers, re.compile(r"education")
            )
            age_col = find_column(headers, re.compile(r"age"))
            liabilities_col = find_column(
                headers, re.compile(r"liabilities")
            )

            for tr in table.find_all("tr"):
                if tr is header_row:
                    continue

                cells = tr.find_all(["td", "th"], recursive=False)
                if len(cells) <= max(candidate_col, assets_col):
                    continue

                candidate_cell = cells[candidate_col]
                anchor = candidate_cell.find(
                    "a", href=CANDIDATE_HREF_RE
                )
                if anchor is None:
                    continue

                name = anchor.get_text(" ", strip=True)
                href = anchor.get("href")
                if not name or not href:
                    continue

                candidate_url = urljoin(url, href)
                candidate_id = parse_qs(
                    urlparse(candidate_url).query
                ).get("candidate_id", [None])[0]

                new_rows.append({
                    "year": year,
                    "state": state,
                    "state_id": page["state_id"],
                    "const": const,
                    "const_id": const_id,
                    "is_byelection": page.get("is_byelection", False),
                    "bye_election_date": page.get("bye_election_date"),
                    "election_date": page.get("election_date"),
                    "candidate": name,
                    "candidate_id": candidate_id,
                    "candidate_url": candidate_url,
                    "is_winner": bool(
                        WINNER_RE.search(
                            candidate_cell.get_text(" ", strip=True)
                        )
                    ),
                    "party": get_cell_value(cells, party_col),
                    "criminal_cases": get_cell_value(cells, criminal_col),
                    "education": get_cell_value(cells, education_col),
                    "age": get_cell_value(cells, age_col),
                    "total_assets": get_cell_value(cells, assets_col),
                    "asset_link": get_image_url(cells[assets_col], url),
                    "liabilities": get_cell_value(cells, liabilities_col),
                    "liabilities_link": (
                        get_image_url(cells[liabilities_col], url)
                        if liabilities_col is not None
                        and liabilities_col < len(cells)
                        else None
                    ),
                    "constituency_url": url,
                    "html_path": str(page["html_path"]),
                })
                page_count += 1

        print("Candidates extracted:", page_count)

        if page_count == 0:
            raise RuntimeError(
                f"No candidates extracted for {year} {state} {const}. "
                "Do not update the CSVs; inspect the rendered page."
            )

        time.sleep(1)

finally:
    driver.quit()

new_candidates = pd.DataFrame(new_rows, columns=columns)

extracted_targets = {
    (int(year), str(state).strip().upper(), str(const_id).strip())
    for year, state, const_id in
    new_candidates[["year", "state", "const_id"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
}

assert extracted_targets == targets, (
    f"Not all target constituencies were extracted. "
    f"Missing: {targets - extracted_targets}"
)

print("\nExtraction successful.")
print("New candidate rows:", len(new_candidates))
display(
    new_candidates.groupby(
        ["year", "state", "const", "const_id"]
    ).size().rename("candidate_rows").reset_index()
)


FileNotFoundError: [Errno 2] No such file or directory: '..\\raw\\lok_sabha_html\\html_manifest.csv'

In [3]:
print(MANIFEST_PATH)

..\raw\lok_sabha_html\html_manifest.csv
